In [28]:
#Import libraries
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score as ss
import pandas as pd 
import numpy as np
import plotly.express as px
import itertools as it 
import folium
from folium.plugins import GroupedLayerControl, FastMarkerCluster
from scipy.spatial import cKDTree

In [29]:
#Import data
nodes = pd.read_csv('data/nodes.csv')
segments = pd.read_csv('data/segments.csv')
segments_status = pd.read_csv('data/segment_status.csv')
streets = pd.read_csv('data/streets.csv')

----
NODES

In [30]:
nodes = pd.read_csv('data/nodes.csv')
nodes

,_id,long,lat
0,366367223,106.629056,10.804243
1,366367233,106.709701,10.771110
2,366367242,106.737189,10.709337
3,366367274,106.760081,10.854489
4,366367285,106.721163,10.804994
...,...,...,...
577962,6202895387,106.647884,10.886330
577963,6202895388,106.649074,10.876678
577964,6203301188,106.700737,10.774919
577965,6203333885,106.699275,10.768892


In [31]:
nodes.info()
nodes.isnull().sum()
nodes.drop_duplicates(inplace=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 577967 entries, 0 to 577966
Data columns (total 3 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   _id     577967 non-null  int64  
 1   long    577967 non-null  float64
 2   lat     577967 non-null  float64
dtypes: float64(2), int64(1)
memory usage: 13.2 MB


----- 
BẢNG SEGMENT


In [32]:
segments = pd.read_csv('data/segments.csv')
segments

,_id,created_at,updated_at,s_node_id,e_node_id,length,street_id,max_velocity,street_level,street_name,street_type
0,0,2020-10-18T13:26:17.365Z,2020-10-18T13:26:17.365Z,373543511,5468660805,114,31096786,80.0,1,Quốc Lộ 1,trunk
1,1,2020-10-18T13:26:17.400Z,2020-10-18T13:26:17.400Z,5468660805,5738158916,9,31096786,80.0,1,Quốc Lộ 1,trunk
2,2,2020-10-18T13:26:17.435Z,2020-10-18T13:26:17.435Z,5738158916,5738158918,23,31096786,80.0,1,Quốc Lộ 1,trunk
3,3,2020-10-18T13:26:17.444Z,2020-10-18T13:26:17.444Z,5738158918,5738158912,66,31096786,80.0,1,Quốc Lộ 1,trunk
4,4,2020-10-18T13:26:17.452Z,2020-10-18T13:26:17.452Z,5738158912,5758104203,127,31096786,80.0,1,Quốc Lộ 1,trunk
...,...,...,...,...,...,...,...,...,...,...,...
84628,84628,2020-10-18T13:30:29.795Z,2020-10-18T13:30:29.795Z,5778600776,411925919,42,658328101,NaN,4,Võ Văn Tần,tertiary
84629,84629,2020-10-18T13:30:29.797Z,2020-10-18T13:30:29.797Z,411925919,3116310151,39,658328101,NaN,4,Võ Văn Tần,tertiary
84630,84630,2020-10-18T13:30:29.799Z,2020-10-18T13:30:29.799Z,3116310151,5778360106,22,658328101,NaN,4,Võ Văn Tần,tertiary
84631,84631,2020-10-18T13:30:29.802Z,2020-10-18T13:30:29.802Z,5778360106,5763168795,37,658328101,NaN,4,Võ Văn Tần,tertiary


In [33]:
segments.drop_duplicates(inplace=True)
segments.info()
segments.isnull().sum()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84633 entries, 0 to 84632
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   _id           84633 non-null  int64  
 1   created_at    84633 non-null  object 
 2   updated_at    84633 non-null  object 
 3   s_node_id     84633 non-null  int64  
 4   e_node_id     84633 non-null  int64  
 5   length        84633 non-null  int64  
 6   street_id     84633 non-null  int64  
 7   max_velocity  9871 non-null   float64
 8   street_level  84633 non-null  int64  
 9   street_name   84481 non-null  object 
 10  street_type   84633 non-null  object 
dtypes: float64(1), int64(6), object(4)
memory usage: 7.1+ MB


_id                 0
created_at          0
updated_at          0
s_node_id           0
e_node_id           0
length              0
street_id           0
max_velocity    74762
street_level        0
street_name       152
street_type         0
dtype: int64

In [34]:
average_max_velocity = segments[segments['street_type'] != 'unclassified'].groupby('street_type')['max_velocity'].mean().reset_index().sort_values('max_velocity', ascending=False)
fig = px.bar(average_max_velocity, x='street_type', y='max_velocity', title='Average Max Velocity by Street Type')
fig.show()

In [35]:
# Create a dictionary from average_max_velocity for easy lookup
velocity_dict = average_max_velocity.set_index('street_type')['max_velocity'].to_dict()
segments['max_velocity'] = segments['max_velocity'].fillna(segments['street_type'].map(velocity_dict))
segments.dropna(subset=['max_velocity'], inplace=True)


-----
BẢNG SEGMENTS_STATUS

In [36]:
segments_status = pd.read_csv('data/segment_status.csv')
segments_status

,_id,updated_at,segment_id,velocity
0,0,2020-07-03T14:55:31.869Z,24845,20
1,1,2020-07-03T15:02:56.048Z,33923,10
2,2,2020-07-04T08:15:52.696Z,33824,5
3,3,2020-07-04T08:15:59.903Z,33824,5
4,4,2020-07-04T08:16:08.201Z,33824,5
...,...,...,...,...
90933,90933,2021-04-22T06:52:39.280Z,52247,1
90934,90934,2021-04-22T06:52:52.501Z,52247,1
90935,90935,2021-04-22T06:53:02.335Z,52247,1
90936,90936,2021-04-22T06:53:14.294Z,52247,1


In [37]:
segments_status.info()
segments_status.isnull().sum()
segments_status.drop_duplicates(inplace=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 90938 entries, 0 to 90937
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   _id         90938 non-null  int64 
 1   updated_at  90938 non-null  object
 2   segment_id  90938 non-null  int64 
 3   velocity    90938 non-null  int64 
dtypes: int64(3), object(1)
memory usage: 2.8+ MB


In [38]:
segments_status['velocity'] = segments_status.groupby('segment_id')['velocity'].transform('mean')
segments_status

,_id,updated_at,segment_id,velocity
0,0,2020-07-03T14:55:31.869Z,24845,16.000000
1,1,2020-07-03T15:02:56.048Z,33923,10.000000
2,2,2020-07-04T08:15:52.696Z,33824,13.000000
3,3,2020-07-04T08:15:59.903Z,33824,13.000000
4,4,2020-07-04T08:16:08.201Z,33824,13.000000
...,...,...,...,...
90933,90933,2021-04-22T06:52:39.280Z,52247,1.350211
90934,90934,2021-04-22T06:52:52.501Z,52247,1.350211
90935,90935,2021-04-22T06:53:02.335Z,52247,1.350211
90936,90936,2021-04-22T06:53:14.294Z,52247,1.350211


----
BẢNG STREETS

In [39]:
streets

,_id,level,max_velocity,name,type
0,31096786,1,80.0,Quốc Lộ 1,trunk
1,32575737,4,NaN,NaN,unclassified
2,32575794,4,NaN,Chu Văn An,unclassified
3,32575820,4,NaN,Nguyễn Văn Bá,tertiary
4,32575823,4,NaN,Nguyễn Thị Nhỏ,tertiary
...,...,...,...,...,...
5548,656562464,4,NaN,NaN,unclassified
5549,656564397,4,NaN,NaN,unclassified
5550,656850719,4,NaN,NaN,unclassified
5551,656851094,4,NaN,NaN,unclassified


--------
CREATING BIG TABLE

In [40]:
big_table = segments.merge(nodes, left_on= ["s_node_id"], right_on = ["_id"], how = "left").merge(segments_status, left_on = ["_id_x"], right_on = ["segment_id"], how = "left")
big_table = big_table.merge(nodes, left_on= ["e_node_id"], right_on = ["_id"], how = "left", suffixes=("_xx", "_yy"))

In [41]:
big_table = big_table[["_id_x", "street_id", "street_name", "street_type", "velocity", "max_velocity", "s_node_id", "long_xx", "lat_xx", "e_node_id", "long_yy", "lat_yy"]]
big_table = big_table.rename(columns={"_id_x": "segment_id", 
                          "long_xx": "s_long", 
                          "lat_xx": "s_lat",
                          "long_yy": "e_long",
                          "lat_yy": "e_lat"})
big_table = big_table.dropna(subset=["s_long", "s_lat", "e_long", "e_lat"])
big_table

,segment_id,street_id,street_name,street_type,velocity,max_velocity,s_node_id,s_long,s_lat,e_node_id,e_long,e_lat
0,0,31096786,Quốc Lộ 1,trunk,NaN,80.000000,373543511,106.601780,10.727718,5468660805,106.601621,10.726701
1,1,31096786,Quốc Lộ 1,trunk,NaN,80.000000,5468660805,106.601621,10.726701,5738158916,106.601607,10.726613
2,2,31096786,Quốc Lộ 1,trunk,NaN,80.000000,5738158916,106.601607,10.726613,5738158918,106.601574,10.726401
3,3,31096786,Quốc Lộ 1,trunk,NaN,80.000000,5738158918,106.601574,10.726401,5738158912,106.601481,10.725809
4,4,31096786,Quốc Lộ 1,trunk,NaN,80.000000,5738158912,106.601481,10.725809,5758104203,106.601277,10.724676
...,...,...,...,...,...,...,...,...,...,...,...,...
142144,84628,658328101,Võ Văn Tần,tertiary,NaN,47.917182,5778600776,106.690870,10.777259,411925919,106.690618,10.776970
142145,84629,658328101,Võ Văn Tần,tertiary,NaN,47.917182,411925919,106.690618,10.776970,3116310151,106.690381,10.776706
142146,84630,658328101,Võ Văn Tần,tertiary,NaN,47.917182,3116310151,106.690381,10.776706,5778360106,106.690243,10.776552
142147,84631,658328101,Võ Văn Tần,tertiary,NaN,47.917182,5778360106,106.690243,10.776552,5763168795,106.690018,10.776299


In [42]:
big_table.info()
big_table.isna().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 142149 entries, 0 to 142148
Data columns (total 12 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   segment_id    142149 non-null  int64  
 1   street_id     142149 non-null  int64  
 2   street_name   142139 non-null  object 
 3   street_type   142149 non-null  object 
 4   velocity      86072 non-null   float64
 5   max_velocity  142149 non-null  float64
 6   s_node_id     142149 non-null  int64  
 7   s_long        142149 non-null  float64
 8   s_lat         142149 non-null  float64
 9   e_node_id     142149 non-null  int64  
 10  e_long        142149 non-null  float64
 11  e_lat         142149 non-null  float64
dtypes: float64(6), int64(4), object(2)
memory usage: 13.0+ MB


segment_id          0
street_id           0
street_name        10
street_type         0
velocity        56077
max_velocity        0
s_node_id           0
s_long              0
s_lat               0
e_node_id           0
e_long              0
e_lat               0
dtype: int64

In [66]:
known_streets = big_table.dropna(subset=['street_name'])
tree = cKDTree(known_streets[['s_lat', 's_long']])

missing_streets = big_table[big_table['street_name'].isna()]
distances, indices = tree.query(missing_streets[['s_lat', 's_long']])

big_table.loc[big_table['street_name'].isna(), 'street_name'] = known_streets.iloc[indices]['street_name'].values

big_table.info()
big_table.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 142149 entries, 0 to 142148
Data columns (total 13 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   segment_id    142149 non-null  int64  
 1   street_id     142149 non-null  int64  
 2   street_name   142149 non-null  object 
 3   street_type   142149 non-null  object 
 4   velocity      142135 non-null  float64
 5   max_velocity  142149 non-null  float64
 6   s_node_id     142149 non-null  int64  
 7   s_long        142149 non-null  float64
 8   s_lat         142149 non-null  float64
 9   e_node_id     142149 non-null  int64  
 10  e_long        142149 non-null  float64
 11  e_lat         142149 non-null  float64
 12  slow_traffic  142149 non-null  int32  
dtypes: float64(6), int32(1), int64(4), object(2)
memory usage: 13.6+ MB


segment_id       0
street_id        0
street_name      0
street_type      0
velocity        14
max_velocity     0
s_node_id        0
s_long           0
s_lat            0
e_node_id        0
e_long           0
e_lat            0
slow_traffic     0
dtype: int64

In [44]:
average_velocity_street_name = big_table.groupby('street_name')['velocity'].mean().reset_index()
average_velocity_street_name = average_velocity_street_name.dropna(subset=['velocity']).sort_values(by='velocity', ascending=False)
fig = px.bar(average_velocity_street_name, x='street_name', y='velocity', title='Average Velocity by Street Name')
fig.show()

In [45]:
big_table['velocity'] = big_table.groupby('street_name')['velocity'].transform(pd.Series.fillna, big_table.groupby('street_name')['velocity'].transform('mean'))
big_table['velocity'] = big_table.groupby('street_type')['velocity'].transform(pd.Series.fillna, big_table.groupby('street_type')['velocity'].transform('mean'))

In [46]:
big_table.loc[big_table['velocity'] > big_table['max_velocity'], 'velocity'] = big_table['max_velocity']
big_table

,segment_id,street_id,street_name,street_type,velocity,max_velocity,s_node_id,s_long,s_lat,e_node_id,e_long,e_lat
0,0,31096786,Quốc Lộ 1,trunk,45.500000,80.000000,373543511,106.601780,10.727718,5468660805,106.601621,10.726701
1,1,31096786,Quốc Lộ 1,trunk,45.500000,80.000000,5468660805,106.601621,10.726701,5738158916,106.601607,10.726613
2,2,31096786,Quốc Lộ 1,trunk,45.500000,80.000000,5738158916,106.601607,10.726613,5738158918,106.601574,10.726401
3,3,31096786,Quốc Lộ 1,trunk,45.500000,80.000000,5738158918,106.601574,10.726401,5738158912,106.601481,10.725809
4,4,31096786,Quốc Lộ 1,trunk,45.500000,80.000000,5738158912,106.601481,10.725809,5758104203,106.601277,10.724676
...,...,...,...,...,...,...,...,...,...,...,...,...
142144,84628,658328101,Võ Văn Tần,tertiary,27.222222,47.917182,5778600776,106.690870,10.777259,411925919,106.690618,10.776970
142145,84629,658328101,Võ Văn Tần,tertiary,27.222222,47.917182,411925919,106.690618,10.776970,3116310151,106.690381,10.776706
142146,84630,658328101,Võ Văn Tần,tertiary,27.222222,47.917182,3116310151,106.690381,10.776706,5778360106,106.690243,10.776552
142147,84631,658328101,Võ Văn Tần,tertiary,27.222222,47.917182,5778360106,106.690243,10.776552,5763168795,106.690018,10.776299


In [47]:
big_table['slow_traffic'] = np.where(big_table['velocity'] >= 20, 0, 1)
big_table

,segment_id,street_id,street_name,street_type,velocity,max_velocity,s_node_id,s_long,s_lat,e_node_id,e_long,e_lat,slow_traffic
0,0,31096786,Quốc Lộ 1,trunk,45.500000,80.000000,373543511,106.601780,10.727718,5468660805,106.601621,10.726701,0
1,1,31096786,Quốc Lộ 1,trunk,45.500000,80.000000,5468660805,106.601621,10.726701,5738158916,106.601607,10.726613,0
2,2,31096786,Quốc Lộ 1,trunk,45.500000,80.000000,5738158916,106.601607,10.726613,5738158918,106.601574,10.726401,0
3,3,31096786,Quốc Lộ 1,trunk,45.500000,80.000000,5738158918,106.601574,10.726401,5738158912,106.601481,10.725809,0
4,4,31096786,Quốc Lộ 1,trunk,45.500000,80.000000,5738158912,106.601481,10.725809,5758104203,106.601277,10.724676,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
142144,84628,658328101,Võ Văn Tần,tertiary,27.222222,47.917182,5778600776,106.690870,10.777259,411925919,106.690618,10.776970,0
142145,84629,658328101,Võ Văn Tần,tertiary,27.222222,47.917182,411925919,106.690618,10.776970,3116310151,106.690381,10.776706,0
142146,84630,658328101,Võ Văn Tần,tertiary,27.222222,47.917182,3116310151,106.690381,10.776706,5778360106,106.690243,10.776552,0
142147,84631,658328101,Võ Văn Tần,tertiary,27.222222,47.917182,5778360106,106.690243,10.776552,5763168795,106.690018,10.776299,0


In [69]:
traffic_coords = pd.concat([big_table[['s_lat', 's_long']].rename(columns={'s_lat': 'lat', 's_long': 'long'}), 
                            big_table[['e_lat', 'e_long']].rename(columns={'e_lat': 'lat', 'e_long': 'long'})], 
                           axis=0).drop_duplicates().reset_index(drop=True)
traffic_coords

,lat,long
0,10.727718,106.601780
1,10.726701,106.601621
2,10.726613,106.601607
3,10.726401,106.601574
4,10.725809,106.601481
...,...,...
43377,10.851948,106.684423
43378,10.837806,106.723628
43379,10.859536,106.650856
43380,10.794735,106.677816


----------
TRAIN 

In [85]:
def loop_dbscan(data, min_clusters, max_clusters):
    best_model = None
    best_score = -1
    best_eps = None
    best_min_samples = None 

    for min_samples in range(3, 10):
        for eps in np.arange(start=0.01, stop=0.0001, step = -0.0005):
            model = DBSCAN(eps=eps, min_samples=min_samples).fit(data)
            labels = model.labels_
            num_clusters = len(set(labels)) - (1 if -1 in labels else 0)
            if num_clusters >= min_clusters and max_clusters >= num_clusters:
                score = ss(data, labels)
                if score > best_score:
                    best_model = model
                    best_score = score
                    best_eps = eps
                    best_min_samples = min_samples
            print("Training with eps =", eps, "and min_samples =", min_samples, "found", num_clusters, "clusters", "with a silhouette score of", score)
                
    return best_model, best_score, best_eps, best_min_samples,labels


In [86]:
loop_dbscan(traffic_coords, 500, 2000)

Training with eps = 0.01 and min_samples = 3 found 545 clusters with a silhouette score of 0.042265327575992544
Training with eps = 0.0095 and min_samples = 3 found 546 clusters with a silhouette score of -0.023693295969997783
Training with eps = 0.009 and min_samples = 3 found 547 clusters with a silhouette score of -0.020946552520473765
Training with eps = 0.008499999999999999 and min_samples = 3 found 547 clusters with a silhouette score of -0.020948270133053837
Training with eps = 0.007999999999999998 and min_samples = 3 found 547 clusters with a silhouette score of -0.021006983230688774
Training with eps = 0.007499999999999998 and min_samples = 3 found 547 clusters with a silhouette score of -0.021006983230688774
Training with eps = 0.0069999999999999975 and min_samples = 3 found 548 clusters with a silhouette score of -0.04007747647530297
Training with eps = 0.006499999999999997 and min_samples = 3 found 549 clusters with a silhouette score of -0.04097582068617379
Training with e

(DBSCAN(eps=0.01, min_samples=4),
 0.13285430793838376,
 0.01,
 4,
 array([ -1,  -1,  -1, ..., 146, 193,  -1], dtype=int64))

In [71]:
model1 = DBSCAN(eps=0.0001, min_samples=4).fit(traffic_coords)
traffic_coords['cluster_labels'] = model1.labels_ 
print(traffic_coords['cluster_labels'].value_counts())
score1 = ss(traffic_coords, traffic_coords['cluster_labels'])
print("Silhouette score:", score1)

cluster_labels
-1      39598
 10        72
 59        67
 192       36
 199       32
        ...  
 388        3
 279        3
 149        3
 145        2
 314        2
Name: count, Length: 544, dtype: int64
Silhouette score: 0.9027279000464391


In [53]:
fig1 = px.scatter(good_traffic_coords_df[['lat', 'long']], 
                        x = "lat",
                        y = "long", 
                        color=good_traffic_coords_df['cluster_labels'].astype(str),
                        labels={"color" : "Cluster"}
                      ) 
fig1

In [54]:
model2 = DBSCAN(0.0009, min_samples=4).fit(slow_traffic_coords_df)
slow_traffic_coords_df['cluster_labels'] = model2.labels_
print(slow_traffic_coords_df['cluster_labels'].value_counts())
score2 = ss(slow_traffic_coords_df, slow_traffic_coords_df['cluster_labels'])
print("Silhouette score:", score2)

cluster_labels
 0      2906
-1       966
 48      303
 28      209
 267     201
        ... 
 362       4
 241       4
 416       4
 470       3
 259       3
Name: count, Length: 475, dtype: int64
Silhouette score: 0.9852349640916165


In [55]:
fig2 = px.scatter(slow_traffic_coords_df[['lat', 'long']], 
                        x = "lat",
                        y = "long", 
                        color=slow_traffic_coords_df['cluster_labels'].astype(str),
                        labels={"color" : "Cluster"}
                      ) 
fig2

map = folium.Map(location=[10.8042433, 106.6290559], zoom_start=10)
for i in range (0, len(nodes)):
   folium.Marker(
      location=[nodes.iloc[i]['lat'], nodes.iloc[i]['long']],
      popup=nodes.iloc[i]['_id'],
   ).add_to(map)
|

map.save('map.html')

-----------------------------------------------------

#Training Model for Traffic Level 2
model2 = DBSCAN(0.005, min_samples=5).fit(cluster2)
TF_2['labels'] = model2.labels_
print(TF_2['labels'].value_counts())
score2 = ss(cluster2, TF_2['labels'])
score2

after_fig2 = px.scatter(TF_2,
                        x = "Latitude" , 
                        y = "Longitude", 
                        color=TF_2['labels'].astype(str),
                        labels={"color" : "Cluster"},
                        )
after_fig2.update_layout(
            title={
            'text' : "Area with traffic level 2",
            'x':0.5,
            'xanchor': 'center',
        })
after_fig2

#Training Model for Traffic Level 3
model3 = DBSCAN(0.005,min_samples=5).fit(cluster3) 
TF_3['labels']= model3.labels_
print(TF_3['labels'].value_counts())
score3 = ss(cluster3, TF_3['labels'])
score3

after_fig3 = px.scatter(TF_3, 
                        x = "Latitude",
                        y = "Longitude", 
                        color=TF_3['labels'].astype(str),
                        labels={"color" : "Cluster"})
after_fig3.update_layout(
            title={
            'text' : "Area with traffic level 3",
            'x':0.5,
            'xanchor': 'center',
        })
after_fig3

#Function to search through Parameters
def getScore(dataset,epss,min_sampless,min_score,min_num_clus):
    if epss == epsilon1:
        cluster = cluster1
    elif epss == epsilon2:
        cluster = cluster2
    elif epss == epsilon3:
        cluster = cluster3
    loop_count = 1
    #Loop through parameters from combinations
    for i, (epsilon, min_sample) in enumerate(list(it.product(epss, min_sampless))):
        #Fit model
        model = DBSCAN(eps = epsilon, min_samples = min_sample).fit(cluster)
        #Get lables
        dataset['labels'] = model.labels_
        num_clus = len(dataset['labels'].value_counts()) 
        #Get scores
        if num_clus > 2:
            score = ss(cluster, dataset['labels'])
            if score >= min_score and num_clus > min_num_clus: 
                print("Loop number:",loop_count,"| Parameters:", epsilon,",", min_sample,"| Score for Trafic Level 1:", score,"| Number of clusters:", num_clus)
        loop_count += 1

#Training Model for Traffic Level 1
model1 = DBSCAN(eps=0.00296,min_samples=7).fit(cluster1)
TF_1['labels'] = model1.labels_
print((TF_1['labels']).value_counts())
score1 = ss(cluster1, TF_1['labels'])
print("Silhouette score:", score1)

#Plot for Model 1
after_fig1 = px.scatter(TF_1, 
                        x = "Latitude",
                        y = "Longitude", 
                        color=TF_1['labels'].astype(str),
                        color_discrete_map={'-1':'red', '0': '#636EFA'},
                        labels={"color" : "Cluster"},
                      ) 
after_fig1.update_layout(
            title={
            'text' : "Area with traffic level 1 with eps = 0.00296, min_samples = 7",
            'x':0.5,
            'xanchor': 'center',
        })


#Training Model for Traffic Level 2
model2 = DBSCAN(0.00481, min_samples=4).fit(cluster2)
TF_2['labels'] = model2.labels_
print(TF_2['labels'].value_counts())
score2 = ss(cluster2, TF_2['labels'])
print("Silhouette score:", score2)

#Plot for Model 2
after_fig2 = px.scatter(TF_2,
                        x = "Latitude" , 
                        y = "Longitude", 
                        color=TF_2['labels'].astype(str),
                        labels={"color" : "Cluster"},
                        )
after_fig2.update_layout(
            title={
            'text' : "Area with traffic level 2 with eps = 0.00481, min_samples = 4",
            'x':0.5,
            'xanchor': 'center',
        })
after_fig2

#Training Model for Traffic Level 3
model3 = DBSCAN(0.0058,min_samples=3).fit(cluster3) 
TF_3['labels']= model3.labels_
print(TF_3['labels'].value_counts())
score3 = ss(cluster3, TF_3['labels'])
print("Silhouette score:", score3)

#Plot for Model 3
after_fig3 = px.scatter(TF_3,
                        x = "Latitude" , 
                        y = "Longitude", 
                        color=TF_3['labels'].astype(str),
                        labels={"color" : "Cluster"},
                        )
after_fig3.update_layout(
            title={
            'text' : "Area with traffic level 3 with eps = 0.0058, min_samples = 3",
            'x':0.5,
            'xanchor': 'center',
        })
after_fig3

#Function to create map 
def createMap(dataset1, dataset2, dataset3):
    m = folium.Map(location=[21.034388, 105.831716], zoom_start=14.5, tiles='cartodbpositron')
    group_tf_1 = folium.FeatureGroup("Traffic Level 1").add_to(m)
    group_tf_2 = folium.FeatureGroup("Traffic Level 2").add_to(m)
    group_tf_3 = folium.FeatureGroup("Traffic Level 3").add_to(m)
    labs1 = dataset1['labels'].to_numpy()
    labs2 = dataset2['labels'].to_numpy()
    labs3 = dataset3['labels'].to_numpy()
    long_lat1 = dataset1.iloc[:,[0,1]].to_numpy()
    long_lat2 = dataset2.iloc[:,[0,1]].to_numpy()
    long_lat3 = dataset3.iloc[:,[0,1]].to_numpy()
    for i in range (0, len(labs1)):
        if labs1[i] == 0:
            folium.CircleMarker(long_lat1[i], radius= 1, color="#636EFA",  fill_opacity=0.9).add_to(group_tf_1)
        if labs1[i] == 1:
            folium.CircleMarker(long_lat1[i], radius= 1, color="#00CC96",  fill_opacity=0.9).add_to(group_tf_1)
        if labs1[i] == 2: 
            folium.CircleMarker(long_lat1[i], radius= 1, color="#AB63FA",  fill_opacity=0.9).add_to(group_tf_1)
        if labs1[i] == -1: 
            folium.CircleMarker(long_lat1[i], radius= 1, color="#EF553B",  fill_opacity=0.9).add_to(group_tf_1)
    for i in range (0, len(labs2)):
        if labs2[i] == 0:
            folium.CircleMarker(long_lat2[i], radius= 1, color="#636EFA",  fill_opacity=0.9).add_to(group_tf_2)
        if labs2[i] == 1:
            folium.CircleMarker(long_lat2[i], radius= 1, color="#00CC96",  fill_opacity=0.9).add_to(group_tf_2)
        if labs2[i] == 2: 
            folium.CircleMarker(long_lat2[i], radius= 1, color="#AB63FA",  fill_opacity=0.9).add_to(group_tf_2)
        if labs2[i] == -1: 
            folium.CircleMarker(long_lat2[i], radius= 1, color="#EF553B",  fill_opacity=0.9).add_to(group_tf_2)
    for i in range (0, len(labs3)):
        if labs3[i] == 0:
            folium.CircleMarker(long_lat3[i], radius= 1, color="#636EFA",  fill_opacity=0.9).add_to(group_tf_3)
        if labs3[i] == 1:
            folium.CircleMarker(long_lat3[i], radius= 1, color="#00CC96",  fill_opacity=0.9).add_to(group_tf_3)
        if labs3[i] == 2: 
            folium.CircleMarker(long_lat3[i], radius= 1, color="#AB63FA",  fill_opacity=0.9).add_to(group_tf_3)
        if labs3[i] == -1: 
            folium.CircleMarker(long_lat3[i], radius= 1, color="#EF553B",  fill_opacity=0.9).add_to(group_tf_3)
    GroupedLayerControl(
        groups={'Traffic Levels': [group_tf_1, group_tf_2,group_tf_3]},
        collapsed=False,
    ).add_to(m)
    folium.LayerControl(collapsed=False)
    return m


createMap(TF_1, TF_2,TF_3)

#FinalMap.save('index.html')
#webbrowser.open('index.html')